# NewsQA 200/11064 benchmark analysis

This notebook reads completed benchmark artifacts only. It does not query Chroma, call a generator, run an LLM judge, or modify the evaluation dataset.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_DIRS = {
    'original_hybrid_noop': PROJECT_ROOT / 'reports/benchmarks/original_hybrid_noop',
    'original_hybrid_crossencoder': PROJECT_ROOT / 'reports/benchmarks/original_hybrid_crossencoder_deepseek',
    'resolved_hybrid_crossencoder': PROJECT_ROOT / 'reports/benchmarks/resolved_hybrid_crossencoder_deepseek',
}
RUN_DIRS

In [ ]:
def load_reports(run_dirs):
    reports = {}
    for name, run_dir in run_dirs.items():
        path = run_dir / 'report.json'
        if path.exists():
            reports[name] = json.loads(path.read_text(encoding='utf-8'))
    return reports

reports = load_reports(RUN_DIRS)
assert reports, 'No report.json files found. Update RUN_DIRS or run the benchmark first.'
list(reports)

In [ ]:
rows = []
for name, report in reports.items():
    rows.append({
        'run': name,
        'success_rate': report.get('coverage', {}).get('success_rate'),
        'hit_rate@5': report.get('retrieval', {}).get('hit_rate@5'),
        'mrr@5': report.get('retrieval', {}).get('mrr@5'),
        'ndcg@5': report.get('retrieval', {}).get('ndcg@5'),
        'delta_mrr@5': report.get('reranker_delta', {}).get('delta_mrr@5'),
        'exact_match': report.get('qa', {}).get('exact_match'),
        'f1': report.get('qa', {}).get('f1'),
        'faithfulness': report.get('ragas', {}).get('faithfulness'),
        'answer_relevancy': report.get('ragas', {}).get('answer_relevancy'),
        'citation_f1': report.get('citations', {}).get('citation_f1'),
        'p95_total_ms': report.get('latency', {}).get('total', {}).get('p95_ms'),
    })
summary = pd.DataFrame(rows).set_index('run')
summary

In [ ]:
metric_columns = ['hit_rate@5', 'mrr@5', 'ndcg@5', 'exact_match', 'f1', 'faithfulness', 'citation_f1']
available = [column for column in metric_columns if summary[column].notna().any()]
ax = summary[available].plot(kind='bar', figsize=(12, 5), ylim=(0, 1))
ax.set_ylabel('Score')
ax.set_title('NewsQA RAG benchmark comparison')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()

In [ ]:
failure_rows = []
for name, report in reports.items():
    for failure in report.get('failures', []):
        failure_rows.append({'run': name, **failure})
pd.DataFrame(failure_rows).head(50)